# GadgetDrop — Sistema de Recomendación de Productos

**Item-Based Collaborative Filtering** sobre datos reales de la tienda.  
Conexión directa a PostgreSQL · Análisis exploratorio · Exportación JSON para la API.

---
## Sección 1 — Importaciones y Conexión

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import psycopg2
import json
from datetime import datetime
from collections import Counter
from sqlalchemy import create_engine, text
from sklearn.metrics.pairwise import cosine_similarity

# Configuración visual
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


In [2]:
# Credenciales — sincronizadas con .env del proyecto
DB_NAME = 'gadgetdrop_db'
DB_USER = 'postgres'
DB_PASSWORD = '16052222771'
DB_HOST = 'localhost'
DB_PORT = 5432

CONNECTION_STRING = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

engine = create_engine(CONNECTION_STRING)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    version = result.fetchone()[0]

print(f'Conexión exitosa a PostgreSQL.')
print(f'Servidor: {version[:60]}...')

Conexión exitosa a PostgreSQL.
Servidor: PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35...


---
## Sección 2 — Carga de Datos

In [3]:
# Query 1: Pedidos con nombre de usuario
query_pedidos = '''
SELECT p.id          AS pedido_id,
       p."usuarioId",
       p.total,
       p.estado,
       p."createdAt",
       u.nombre      AS usuario_nombre
FROM   pedidos  p
JOIN   usuarios u ON p."usuarioId" = u.id
'''

pedidos_df = pd.read_sql(query_pedidos, engine)
pedidos_df['createdAt'] = pd.to_datetime(pedidos_df['createdAt'])
pedidos_df['total']     = pedidos_df['total'].astype(float)

print(f'pedidos_df  → shape: {pedidos_df.shape}')
pedidos_df.head()

pedidos_df  → shape: (181, 6)


,pedido_id,usuarioId,total,estado,createdAt,usuario_nombre
0,1,1,400.00,Enviado,2025-05-22 06:20:09.737000+00:00,Carlos Mendoza
1,2,3,299.95,pendiente,2025-05-22 18:34:52.497000+00:00,Juan Diego Portilla Riveros
2,3,4,178839.96,pendiente,2025-05-26 16:06:48.120000+00:00,David Rodríguez
3,4,4,178839.96,pendiente,2025-05-26 16:06:48.124000+00:00,David Rodríguez
4,5,4,120150.00,pendiente,2025-05-26 17:54:01.729000+00:00,David Rodríguez


In [4]:
# Query 2: Detalles de pedido con nombre y categoría de producto
query_detalles = '''
SELECT dp.id,
       dp."pedidoId",
       dp."productoId",
       dp.cantidad,
       dp."precioUnitario",
       pr.nombre    AS producto_nombre,
       pr.categoria,
       pr.precio
FROM   detalle_pedidos dp
JOIN   productos       pr ON dp."productoId" = pr.id
'''

detalles_df = pd.read_sql(query_detalles, engine)
detalles_df['precioUnitario'] = detalles_df['precioUnitario'].astype(float)
detalles_df['precio']         = detalles_df['precio'].astype(float)
detalles_df['ingresos']       = detalles_df['cantidad'] * detalles_df['precioUnitario']

print(f'detalles_df → shape: {detalles_df.shape}')
detalles_df.head()

detalles_df → shape: (476, 9)


,id,pedidoId,productoId,cantidad,precioUnitario,producto_nombre,categoria,precio,ingresos
0,2,2,3,5,59.99,"Powerbank 20,000mAh Carga Rápida",Accesorios Móviles,89.99,299.95
1,3,3,2,4,150.00,Anillo de Luz LED 26cm,Creadores de Contenido,49.99,600.00
2,4,4,2,4,150.00,Anillo de Luz LED 26cm,Creadores de Contenido,49.99,600.00
3,5,3,3,4,59.99,"Powerbank 20,000mAh Carga Rápida",Accesorios Móviles,89.99,239.96
4,6,4,3,4,59.99,"Powerbank 20,000mAh Carga Rápida",Accesorios Móviles,89.99,239.96


In [ ]:
# Query 3: Catálogo completo de productos
query_productos = '''
SELECT id, nombre, precio, stock, categoria
FROM   productos
ORDER  BY categoria, nombre
'''

productos_df = pd.read_sql(query_productos, engine)
productos_df['precio'] = productos_df['precio'].astype(float)

print(f'productos_df → shape: {productos_df.shape}')
productos_df.head()

productos_df → shape: (24, 5)


,id,nombre,precio,stock,categoria
0,7,Cable USB-C a USB-C 60W,29.99,80,Accesorios Móviles
1,32,Cargador Inalámbrico 15W,34.99,60,Accesorios Móviles
2,16,Estuche Organizador para Gadgets,14.99,70,Accesorios Móviles
3,33,"Funda Rígida para Laptop 15""",24.99,45,Accesorios Móviles
4,3,"Powerbank 20,000mAh Carga Rápida",89.99,35,Accesorios Móviles


: 

---
## Sección 3 — Análisis Exploratorio (EDA)

### 3.1 Resumen General del Negocio

In [ ]:
# Filtro de pedidos activos (excluye cancelados para ingresos)
estados_activos = ['entregado', 'enviado']
pedidos_activos = pedidos_df[pedidos_df['estado'].isin(estados_activos)]
detalles_activos = detalles_df[detalles_df['pedidoId'].isin(pedidos_activos['pedido_id'])]

total_usuarios   = pedidos_df['usuarioId'].nunique()
total_pedidos    = len(pedidos_df)
total_vendidos   = detalles_df['cantidad'].sum()
ingresos_totales = detalles_activos['ingresos'].sum()
ticket_promedio  = pedidos_activos['total'].mean()
prod_mas_vendido = (
    detalles_df.groupby('producto_nombre')['cantidad']
    .sum().idxmax()
)
cat_mas_vendida  = (
    detalles_df.groupby('categoria')['ingresos']
    .sum().idxmax()
)

print('=' * 50)
print('  RESUMEN GENERAL — GadgetDrop')
print('=' * 50)
print(f'  Usuarios con compras     : {total_usuarios}')
print(f'  Total de pedidos         : {total_pedidos}')
print(f'  Unidades vendidas        : {total_vendidos}')
print(f'  Ingresos totales         : ${ingresos_totales:,.2f}')
print(f'  Ticket promedio          : ${ticket_promedio:,.2f}')
print(f'  Producto más vendido     : {prod_mas_vendido}')
print(f'  Categoría más vendida    : {cat_mas_vendida}')
print('=' * 50)

### 3.2 Gráfica 1 — Evolución de Ventas Mensuales

In [ ]:
ventas_mes = (
    pedidos_activos
    .assign(mes=pedidos_activos['createdAt'].dt.to_period('M'))
    .groupby('mes')
    .agg(cantidad_pedidos=('pedido_id', 'count'), ingresos=('total', 'sum'))
    .reset_index()
    .sort_values('mes')
)
ventas_mes['mes_str'] = ventas_mes['mes'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

bars = ax1.bar(ventas_mes['mes_str'], ventas_mes['cantidad_pedidos'],
               color='steelblue', alpha=0.75, label='Pedidos')
line = ax2.plot(ventas_mes['mes_str'], ventas_mes['ingresos'],
                color='darkorange', marker='o', linewidth=2.5, label='Ingresos ($)')

ax1.set_xlabel('Mes')
ax1.set_ylabel('Cantidad de Pedidos', color='steelblue')
ax2.set_ylabel('Ingresos ($)', color='darkorange')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Evolución de Ventas Mensuales - GadgetDrop', fontsize=14, pad=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3.3 Gráfica 2 — Top 10 Productos Más Vendidos

In [ ]:
top10 = (
    detalles_df.groupby(['producto_nombre', 'categoria'])['cantidad']
    .sum()
    .reset_index()
    .sort_values('cantidad', ascending=False)
    .head(10)
)

categorias_unicas = top10['categoria'].unique()
palette = dict(zip(categorias_unicas, sns.color_palette('tab10', len(categorias_unicas))))
colores = top10['categoria'].map(palette)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10['producto_nombre'], top10['cantidad'],
               color=colores, edgecolor='white', height=0.65)

for bar, val in zip(bars, top10['cantidad']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(int(val)), va='center', fontsize=10)

# Leyenda de categorías
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=palette[c], label=c) for c in categorias_unicas]
ax.legend(handles=legend_elements, title='Categoría', loc='lower right')

ax.set_xlabel('Unidades Vendidas')
ax.invert_yaxis()
plt.title('Top 10 Productos Más Vendidos', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### 3.4 Gráfica 3 — Distribución de Ingresos por Categoría

In [ ]:
ingresos_cat = (
    detalles_activos.groupby('categoria')['ingresos']
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    ingresos_cat,
    labels=ingresos_cat.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set2', len(ingresos_cat)),
    pctdistance=0.82,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(11)

plt.title('Distribución de Ingresos por Categoría', fontsize=14, pad=16)
plt.tight_layout()
plt.show()

### 3.5 Gráfica 4 — Segmentación de Usuarios por Comportamiento de Compra

In [ ]:
usuario_stats = (
    pedidos_df.groupby(['usuarioId', 'usuario_nombre'])
    .agg(
        num_pedidos=('pedido_id', 'count'),
        gasto_total=('total', 'sum'),
        ticket_prom=('total', 'mean')
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    usuario_stats['num_pedidos'],
    usuario_stats['gasto_total'],
    s=usuario_stats['ticket_prom'] * 4,
    c=usuario_stats['num_pedidos'],
    cmap='viridis',
    alpha=0.8,
    edgecolors='white',
    linewidths=0.8
)

for _, row in usuario_stats.iterrows():
    ax.annotate(
        row['usuario_nombre'].split()[0],
        (row['num_pedidos'], row['gasto_total']),
        fontsize=8, alpha=0.75,
        xytext=(4, 4), textcoords='offset points'
    )

plt.colorbar(scatter, ax=ax, label='Nº de Pedidos')
ax.set_xlabel('Número de Pedidos')
ax.set_ylabel('Gasto Total ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.title('Segmentación de Usuarios por Comportamiento de Compra', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### 3.6 Gráfica 5 — Heatmap de Ingresos por Categoría y Mes

In [ ]:
# Unir detalles activos con fecha del pedido
det_fecha = detalles_activos.merge(
    pedidos_activos[['pedido_id', 'createdAt']],
    left_on='pedidoId', right_on='pedido_id'
)
det_fecha['mes'] = det_fecha['createdAt'].dt.to_period('M').astype(str)

heatmap_data = (
    det_fecha.groupby(['categoria', 'mes'])['ingresos']
    .sum()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Ingresos ($)'}
)
ax.set_xlabel('Mes')
ax.set_ylabel('Categoría')
plt.title('Mapa de Calor — Ingresos por Categoría y Mes', fontsize=14, pad=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Sección 4 — Modelo de Recomendación (Item-Based Collaborative Filtering)

### 4.1 Construcción de la Matriz Usuario-Producto

In [ ]:
# Unir detalles con pedidos para obtener el usuarioId
det_usuario = detalles_df.merge(
    pedidos_df[['pedido_id', 'usuarioId']],
    left_on='pedidoId', right_on='pedido_id'
)

# Pivot: filas = usuario, columnas = producto, valores = cantidad total comprada
usuario_producto = (
    det_usuario
    .groupby(['usuarioId', 'productoId'])['cantidad']
    .sum()
    .unstack(fill_value=0)
)

print(f'Matriz usuario-producto: {usuario_producto.shape}')
print(f'  → {usuario_producto.shape[0]} usuarios × {usuario_producto.shape[1]} productos')
usuario_producto.head()

### 4.2 Similitud entre Productos (cosine similarity)

In [ ]:
# Transponer: productos x usuarios para calcular similitud entre productos
producto_usuario = usuario_producto.T

similitud = cosine_similarity(producto_usuario)

# Mapa id → nombre de producto
id_a_nombre = dict(zip(productos_df['id'], productos_df['nombre']))
id_a_cat    = dict(zip(productos_df['id'], productos_df['categoria']))
id_a_precio = dict(zip(productos_df['id'], productos_df['precio']))

nombres_cols = [id_a_nombre.get(pid, str(pid)) for pid in producto_usuario.index]

similitud_df = pd.DataFrame(
    similitud,
    index=nombres_cols,
    columns=nombres_cols
)

print(f'Matriz de similitud: {similitud_df.shape}')
similitud_df.head()

### 4.3 Función de Recomendación

In [ ]:
# Mapas inversos necesarios para enriquecer resultados
nombre_a_id = {v: k for k, v in id_a_nombre.items()}

def recomendar_productos(producto_nombre, n=5):
    """
    Dado el nombre de un producto, retorna los n productos
    más similares con su score de similitud.
    Excluye el producto mismo.
    Retorna DataFrame con columnas: producto, categoria, similitud, precio.
    """
    if producto_nombre not in similitud_df.index:
        print(f'Producto "{producto_nombre}" no encontrado en la matriz.')
        return pd.DataFrame()

    scores = (
        similitud_df[producto_nombre]
        .drop(labels=[producto_nombre])     # excluye el mismo
        .sort_values(ascending=False)
        .head(n)
    )

    resultado = pd.DataFrame({
        'producto'  : scores.index,
        'similitud' : scores.values,
    })
    resultado['categoria'] = resultado['producto'].map(
        lambda n: id_a_cat.get(nombre_a_id.get(n))
    )
    resultado['precio'] = resultado['producto'].map(
        lambda n: id_a_precio.get(nombre_a_id.get(n))
    )
    return resultado.reset_index(drop=True)

print('Función recomendar_productos() definida.')

### 4.4 Pruebas del Modelo

In [ ]:
productos_prueba = [
    'Teclado Mecánico RGB',
    'Micrófono Condensador',
    'Smartwatch Deportivo Serie 8',
    'Powerbank 20,000mAh Carga Rápida',
]

for nombre in productos_prueba:
    print(f'\n{'─' * 55}')
    print(f'  Recomendaciones para: "{nombre}"')
    print(f'{'─' * 55}')
    resultado = recomendar_productos(nombre, n=5)
    if not resultado.empty:
        resultado['similitud'] = resultado['similitud'].round(4)
        resultado['precio']    = resultado['precio'].apply(lambda x: f'${x:.2f}')
        print(resultado.to_string(index=False))

### 4.5 Gráfica 6 — Heatmap de Similitud entre Productos

In [ ]:
# Top 12 productos más vendidos
top12_nombres = (
    detalles_df.groupby('producto_nombre')['cantidad']
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .index.tolist()
)

# Filtrar solo los que estén en la matriz de similitud
top12_en_matriz = [n for n in top12_nombres if n in similitud_df.index]
sub_similitud   = similitud_df.loc[top12_en_matriz, top12_en_matriz]

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    sub_similitud,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    linewidths=0.4,
    vmin=0, vmax=1,
    ax=ax,
    cbar_kws={'label': 'Similitud coseno'}
)
plt.title('Matriz de Similitud entre Productos — Modelo de Recomendación',
          fontsize=13, pad=14)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

---
## Sección 5 — Exportar Recomendaciones para la API

In [ ]:
import os

recomendaciones_json = {
    'generado_en'    : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_productos': len(producto_usuario.index),
    'recomendaciones': {}
}

for prod_id in producto_usuario.index:
    nombre = id_a_nombre.get(prod_id, str(prod_id))
    if nombre not in similitud_df.index:
        continue

    top5 = (
        similitud_df[nombre]
        .drop(labels=[nombre])
        .sort_values(ascending=False)
        .head(5)
    )

    recomendados = []
    for rec_nombre, score in top5.items():
        rec_id = nombre_a_id.get(rec_nombre)
        if rec_id is None:
            continue
        recomendados.append({
            'productoId': int(rec_id),
            'nombre'    : rec_nombre,
            'categoria' : id_a_cat.get(rec_id),
            'precio'    : float(id_a_precio.get(rec_id, 0)),
            'similitud' : round(float(score), 4)
        })

    recomendaciones_json['recomendaciones'][str(prod_id)] = {
        'nombre'       : nombre,
        'categoria'    : id_a_cat.get(prod_id),
        'recomendados' : recomendados
    }

output_path = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                           'data_science', 'recomendaciones.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(recomendaciones_json, f, ensure_ascii=False, indent=2)

total_exp = len(recomendaciones_json['recomendaciones'])
print(f'Archivo exportado con {total_exp} productos y sus recomendaciones')
print(f'Ruta: {output_path}')

---
## Sección 6 — Conclusiones

### Modelo Implementado

Se implementó un sistema de recomendación basado en **Item-Based Collaborative Filtering** usando **similitud coseno** sobre la matriz de interacciones usuario-producto. El modelo identifica qué productos tienden a ser comprados por los mismos usuarios y los recomienda como complementarios.

### Métricas Obtenidas

| Métrica | Valor |
|---|---|
| Productos analizados | 24 |
| Usuarios únicos con compras | 24 |
| Pedidos generados | ~114 |
| Detalles de compra | ~353 |
| Categorías | 5 (Gaming, Workstation, Creadores, Accesorios Móviles, Wearables) |

### Limitaciones con Datos Simulados

- Los patrones de compra fueron **diseñados intencionalmente**, lo que hace que la similitud coseno refleje la estructura del seed y no comportamiento real de usuarios.
- La **densidad de la matriz** es artificialmente alta: en producción real, la mayoría de usuarios habrán comprado muy pocos productos (problema de *cold start*).
- Los pesos de los patrones crean correlaciones fuertes entre productos de la misma categoría, que pueden no reproducirse con tráfico orgánico.

### Propuesta de Mejora con Datos Reales

1. **Matrix Factorization (SVD/ALS):** mejora la precisión con matrices dispersas de producción.
2. **Modelo híbrido:** combinar filtrado colaborativo con similitud de contenido (categoría, precio, descripción via embeddings).
3. **A/B testing:** evaluar CTR y conversión de las recomendaciones en la UI.
4. **Reentrenamiento automático:** cron job semanal que regenere `recomendaciones.json` con los pedidos más recientes.
5. **Cold start:** para usuarios nuevos, recomendar los top-sellers de la categoría más vista en sesión.

### Tecnologías Usadas

| Capa | Tecnología |
|---|---|
| Base de datos | PostgreSQL + SQLAlchemy + psycopg2 |
| Manipulación de datos | pandas · numpy |
| Visualización | matplotlib · seaborn |
| Modelo ML | scikit-learn (cosine_similarity) |
| Exportación | JSON estándar → consumible por la API Express |
| Entorno | Jupyter Notebook |
